# RSNA Knee Abnormality Detection â€” Colab pipeline

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vahid-karami/rsna-knee-mri/blob/main/notebooks/rsna_knee_colab.ipynb)

Detect 12 knee abnormalities (ACL/MCL tears, meniscal tears, osteoarthritis, effusion, synovitis, Baker's cyst, contusion, fracture) on MRI.

Only 58 of ~4,400 training studies have expert labels, so the pipeline **learns from the radiology reports**:

1. **Report labelling** â€” a rule-based multilingual extractor + an open LLM (Qwen2.5) turn the free-text reports into soft labels, checked against the expert labels.
2. **Image model** â€” a 2.5D multi-view CNN (sagittal / coronal / axial) trained on those soft labels.
3. **Honest evaluation** â€” macro ROC-AUC (the competition metric) on the expert-labelled studies, which are never used for training or model selection.

**Before you start:** `Runtime â†’ Change runtime type â†’ T4 GPU`. Then run the cells from top to bottom.

## 1. Setup

In [ ]:
REPO = "https://github.com/vahid-karami/rsna-knee-mri.git"
!rm -rf /content/rsna-knee-mri && git clone -q $REPO /content/rsna-knee-mri
%cd /content/rsna-knee-mri
!pip install -q -r requirements.txt

Outputs go to Google Drive, so a disconnected Colab session loses nothing (the LLM labelling step resumes where it stopped).

`run(...)` executes a pipeline step (e.g. `run("knee.build_labels")`) inside this notebook, with the same arguments as the command line.

In [ ]:
import os, sys, gc, runpy
from google.colab import drive
drive.mount("/content/drive")

SRC = "/content/rsna-knee-mri/src"
os.environ["KNEE_DATA_ROOT"] = "/content/data"
os.environ["KNEE_WORK_DIR"]  = "/content/drive/MyDrive/rsna-knee/work"
os.environ["KNEE_CACHE_DIR"] = "/content/cache"          # local disk is faster than Drive for training
os.environ["PYTHONPATH"] = SRC + os.pathsep + os.environ.get("PYTHONPATH", "")
os.makedirs(os.environ["KNEE_WORK_DIR"], exist_ok=True)
if SRC not in sys.path:
    sys.path.insert(0, SRC)


def run(module, *args):
    """Run `python -m <module> <args>` inside this notebook."""
    for name in [m for m in sys.modules if m == "knee" or m.startswith("knee.")]:
        del sys.modules[name]  # always use the latest code from the repo
    argv = sys.argv
    sys.argv = [module, *map(str, args)]
    try:
        runpy.run_module(module, run_name="__main__")
    finally:
        sys.argv = argv
        gc.collect()
        try:
            import torch; torch.cuda.empty_cache()
        except Exception:
            pass


import knee  # fails here, not later, if the code is missing
print("knee package loaded from", knee.__file__)

### Kaggle API token
Paste your token (kaggle.com/settings â†’ API â†’ Generate New Token) between the quotes below.
If you leave it empty, the cell uses a Colab secret named `KAGGLE_API_TOKEN` if you made one, otherwise it asks you for the token.

âš ï¸ This repo is public: after pasting, don't use *File â†’ Save a copy in GitHub*, or the token is published.

In [ ]:
import getpass, pathlib

KAGGLE_API_TOKEN = ""   # <- paste your token here

if not KAGGLE_API_TOKEN:
    try:
        from google.colab import userdata
        KAGGLE_API_TOKEN = userdata.get("KAGGLE_API_TOKEN") or ""
    except Exception:
        pass
if not KAGGLE_API_TOKEN:
    KAGGLE_API_TOKEN = getpass.getpass("Paste your Kaggle API token: ").strip()

os.environ["KAGGLE_API_TOKEN"] = KAGGLE_API_TOKEN
cfg = pathlib.Path.home() / ".kaggle"
cfg.mkdir(exist_ok=True)
(cfg / "access_token").write_text(KAGGLE_API_TOKEN)
(cfg / "access_token").chmod(0o600)

!pip install -q -U kaggle
!kaggle competitions list -s knee   # a table here = credentials OK

## 2. Download the tables (reports, labels, series metadata: a few MB, not the 250 GB of images)

In [ ]:
%%bash
mkdir -p /content/data && cd /content/data
for f in train.csv train_series.csv test.csv test_series.csv sample_submission.csv; do
  [ -s "$f" ] || kaggle competitions download -c rsna-knee-abnormality-detection -f $f -p . -q || echo "could not get $f"
done
for z in *.zip; do [ -e "$z" ] && unzip -oq "$z" && rm "$z"; done
ls -lh

## 3. Explore the data

In [ ]:
import json, glob, io, subprocess
import pandas as pd, numpy as np, matplotlib.pyplot as plt
from knee import config as C
from knee.data import load_train, load_series, gold_mask

labels = C.load_label_names()
train = load_train(); series = load_series("train")
gold = gold_mask(train, labels)
print(f"{len(train)} studies | {len(series)} series | {gold.sum()} expert-labelled studies")
print(series[C.PLANE_COL].value_counts())
train[gold][labels].mean().sort_values().plot.barh(title="Prevalence in expert-labelled studies", figsize=(6, 4));

In [ ]:
print(train[C.REPORT_COL].str.len().describe())
for r in train[C.REPORT_COL].sample(2, random_state=0):
    print("-" * 80); print(r[:1200])

Which languages are the reports written in?

In [ ]:
!pip install -q langdetect
from langdetect import detect, DetectorFactory
DetectorFactory.seed = 0

def lang(text):
    try:
        return detect(str(text)[:600])
    except Exception:
        return "unknown"

train["lang"] = train[C.REPORT_COL].map(lang)
train["lang"].value_counts()

## 4. Labels from reports
### 4a. Rule-based baseline (seconds)
The table shows how well each label source agrees with the 58 expert-labelled studies (ROC-AUC, 1.0 = perfect).

In [ ]:
run("knee.build_labels")

### 4b. LLM labeller (Qwen2.5-3B-Instruct on the T4)
First 32 reports, to check speed and output:

In [ ]:
run("knee.report_llm", "--limit", 32, "--batch-size", 16)

In [ ]:
answers = f"{os.environ['KNEE_WORK_DIR']}/labels/llm_answers.jsonl"
with open(answers) as f:
    rows = [json.loads(line) for line in f]
print(f"{len(rows)} answers so far, {sum(r['labels'] is None for r in rows)} unparsable")
for r in rows[:2]:
    print(r["answer"][:400], "\n")

Now all reports: about 1-1.5 h on a T4. If Colab disconnects, run the **Setup** and **token** cells again, then this cell: it resumes where it stopped.
For better labels with more time, add `"--model", "Qwen/Qwen2.5-7B-Instruct"` and use batch size 8.

In [ ]:
run("knee.report_llm", "--batch-size", 16)

In [ ]:
run("knee.build_labels")   # compares rules vs LLM vs both against the experts and keeps the best

## 5. MRI images
The full image set (~250 GB) does not fit on Colab, so it is preprocessed **once on Kaggle** (where the data is mounted) into a ~4 GB cache of 1,500 studies.

1. On Kaggle: *Create â†’ New Notebook* â†’ *File â†’ Import Notebook* â†’ paste
   `https://github.com/vahid-karami/rsna-knee-mri/blob/main/notebooks/kaggle_preprocess.ipynb`.
2. Right panel: *Add Input* â†’ **RSNA Knee Abnormality Detection**; *Session options* â†’ **Internet on**.
3. Rename the notebook to **rsna-knee-preprocess**, then *Save Version â†’ Save & Run All (Commit)*.
4. When the version shows **Complete** (about 1-2 h), run the next cell.

In [ ]:
def find_my_kernel(name="rsna-knee-preprocess"):
    out = subprocess.run(["kaggle", "kernels", "list", "--mine", "--search", name, "--csv"],
                         capture_output=True, text=True).stdout.splitlines()
    start = next((i for i, line in enumerate(out) if line.startswith("ref")), None)  # skip CLI warnings
    if start is None:
        return None
    refs = pd.read_csv(io.StringIO("\n".join(out[start:])))["ref"].astype(str)
    refs = [r.split("kaggle.com/code/")[-1].strip("/") for r in refs]
    return next((r for r in refs if r.endswith("/" + name)), refs[0] if refs else None)

kernel = find_my_kernel()
if kernel is None:
    raise SystemExit("No Kaggle notebook named 'rsna-knee-preprocess' found in your account. Do steps 1-3 above first.")
print("Downloading output of", kernel)
!rm -rf /content/cache_dl && mkdir -p /content/cache_dl /content/cache
!kaggle kernels output {kernel} -p /content/cache_dl
if not os.path.exists("/content/cache_dl/knee_cache.tar"):
    raise SystemExit("knee_cache.tar is not in the output yet. Wait until the Kaggle run shows 'Complete', then re-run this cell.")
!tar -xf /content/cache_dl/knee_cache.tar -C /content/cache && rm -rf /content/cache_dl
print(len(glob.glob("/content/cache/*.npz")), "studies in the cache")

In [ ]:
files = sorted(glob.glob("/content/cache/*.npz"))
assert files, "The image cache is empty: run the cell above first."
d = np.load(files[0]); vol = d["vol"]
fig, ax = plt.subplots(1, 3, figsize=(12, 4))
for v, name in enumerate(C.VIEWS):
    ax[v].imshow(vol[v, vol.shape[1] // 2], cmap="gray"); ax[v].set_title(name); ax[v].axis("off")

## 6. Train the image model (about 5-8 min per epoch on a T4 for 1,500 studies)
If the GPU runs out of memory, add `"--grad-checkpointing"` or use `"--slices", 12`.

In [ ]:
run("knee.train_image", "--epochs", 10, "--batch-size", 4, "--accum", 2,
    "--slices", 16, "--backbone", "efficientnet_b0")

## 7. Results

In [ ]:
res = json.load(open(f"{os.environ['KNEE_WORK_DIR']}/models/results.json"))
h = pd.DataFrame(res["history"])
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
ax[0].plot(h.epoch, h.val_auc, "o-", label="held-out (report labels)")
ax[0].plot(h.epoch, h.gold_auc, "s-", label="expert labels")
ax[0].set(xlabel="epoch", ylabel="macro ROC-AUC", title="Validation"); ax[0].legend(); ax[0].grid(alpha=.3)
pl = pd.Series(res["gold_per_label_auc"]).dropna().sort_values()
ax[1].barh(pl.index, pl.values, color="#3b6ea5"); ax[1].axvline(.5, ls="--", c="gray")
ax[1].set(xlim=(0, 1), title=f"Per-finding AUC on expert labels (macro {res['gold_macro_auc']:.3f})")
plt.tight_layout(); plt.savefig(f"{os.environ['KNEE_WORK_DIR']}/results.png", dpi=150)
print({k: res[k] for k in ("best_val_auc", "gold_macro_auc", "n_train", "n_gold")})

In [ ]:
pd.read_csv(f"{os.environ['KNEE_WORK_DIR']}/labels/label_agreement.csv", index_col=0).round(3)

## 8. Next steps
* Train several models with different seeds or backbones (`"--backbone", "convnext_nano"`) and average them.
* Preprocess all ~4,400 studies (`MAX_STUDIES = 0`) on Kaggle.
* **Late submission:** upload `best.pt` as a Kaggle dataset, attach it and this repo to a Kaggle notebook (internet off), and run `python -m knee.predict --checkpoints /kaggle/input/<dataset>/*.pt`.